# 06n — Generalization: **frequency-controlled** order test (Phase 2)

**The question.** Does symbol *order* ever carry class signal beyond symbol *frequency*, on **real** data?

**Why this notebook exists.** `06m` ran the §15 ΔAUC order-null on Breakfast's top-6 activities and got
`order_helps` **0/60** — but with `auc_intact = 1.0` on *all 60 cells*. The activities have near-disjoint
action vocabularies, so a bare unigram histogram already separates them perfectly. With `AUC_intact` pinned
at the ceiling, `delta_auc` is 0 **by arithmetic**: the null could not have fired even if order carried
signal. **That 0/60 is vacuous, not evidence** — and the same doubt hangs over SDS2's 0/15.

So the headline deliverable here is **the screen itself**: for every cached real task, measure whether the
null had any room to fire, and only then run it where it did.

**The instrument.** `headroom_table` reports the out-of-fold **unigram-histogram AUC** — the exact frequency
channel the shuffle holds fixed — and assigns a **two-sided** band. `hist_auc < 1.0` is necessary but *not*
sufficient:

| band | meaning | is a null informative? |
|---|---|---|
| `saturated` (`hist_auc ≥ 0.95`) | frequency already separates the classes | **No** — no room. The 06m failure. |
| `floor` (`hist_auc ≤ 0.55`) | frequency separates nothing | **No** — "order doesn't help" is indistinguishable from "these classes are exchangeable". The same failure, mirrored. |
| `sweet` | distinguishable *and* not saturated | **Yes** |

**Gating on `hist_auc` is legitimate.** It is a deterministic function of shuffle-*invariant* quantities (a
sequence's histogram *is* its multiset; labels and splits are untouched by the shuffle), so it is a
*conditioning variable*, not a statistic — no selection bias enters. Pinned as executable invariants in
`tests/test_headroom.py`.

**Bounds (reported honestly).** Breakfast is **deduplicated to one view per execution** (1712 → 503) before
any CV. The order-null runs only on `sweet`-band tasks. The primary family is **one pre-registered combo**
at `n_shuffles=1000`; the full 2×2 grid at 200 is reported as secondary. A **calibration decoy** measures
the probe's empirical type-I rate before the real null is run. **A null is a fine result — no positive is
chased.** Kernel: `smartflat_repro`.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault('NUMBA_THREADING_LAYER', 'workqueue')  # fork-safe rTWE under nbconvert
os.environ.setdefault('MPLBACKEND', 'agg')                   # headless figures
import collections, hashlib
import numpy as np, pandas as pd
from collections import Counter
from IPython.display import display

from smartflat.utils.utils_io import get_data_root
from smartflat.features.symbolic_barycenter.generalization.action_segmentation import (
    load_action_seg, dedup_by_execution, trial_labels, parse_video_id,
    build_action_seg_ground_cost, default_root, read_mapping, DATASETS)
from smartflat.features.symbolic_barycenter.generalization.headroom import (
    headroom_table, restrict_to_shared_vocabulary, pooled_markov_surrogate)
from smartflat.features.symbolic_barycenter.generalization.suite import run_generalization_suite
from smartflat.features.symbolic_barycenter.order_evaluation import order_information
from smartflat.features.symbolic_barycenter.evaluation import evaluate_incremental_ordering

NAME = 'breakfast'
GEN = os.path.join(get_data_root(), 'outputs', 'symbolic_barycenter', 'generalization')
OUT = os.path.join(GEN, 'frequency_controlled')
os.makedirs(OUT, exist_ok=True)

# 06m's committed CSVs feed the paper (§6.5/§7.2). Pin them now, re-check in the last cell:
# this notebook must never write into their directory.
BF_DIR = os.path.join(GEN, NAME)
assert OUT != BF_DIR, 'OUT must not be 06m output dir'
FROZEN = {p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
          for p in (os.path.join(BF_DIR, 'order_null.csv'),
                    os.path.join(BF_DIR, 'quality.csv')) if os.path.isfile(p)}
print('output dir:', OUT)
for p, h in FROZEN.items():
    print('  pinned 06m %-14s sha256=%s' % (os.path.basename(p), h[:16]))

## 1. Load + deduplicate to distinct executions

Breakfast films each execution with up to five cameras **simultaneously**, and the frame-level ground truth
of those views is the *same annotation*. They are not extra data — they are copies. The §15 harness's
`RepeatedStratifiedKFold` is not group-aware, so leaving them in places copies of one execution in train
*and* test, letting a classifier recognise rather than generalise. Everything below runs on the
deduplicated set.

In [ ]:
meta_raw, X_raw, labels_raw, G = load_action_seg(NAME)
meta, X, labels, keep_idx = dedup_by_execution(meta_raw, X_raw, labels_raw, NAME)
print('%s: %d videos -> %d distinct executions (%d subjects)'
      % (NAME, len(X_raw), len(X), len({parse_video_id(NAME, os.path.splitext(v)[0])['subject']
                                        for v in meta_raw['video']})))

# Evidence that the discarded views are duplicates, not data.
groups = collections.defaultdict(list)
for i, v in enumerate(meta_raw['video']):
    d = parse_video_id(NAME, os.path.splitext(v)[0])
    groups[(d['subject'], d['activity'])].append(i)
same = tot = 0; jac = []
for ix in groups.values():
    base = tuple(X_raw[ix[0]].tolist())
    for j in ix[1:]:
        tot += 1; o = tuple(X_raw[j].tolist()); same += (o == base)
        a, b = set(base), set(o); jac.append(len(a & b) / len(a | b))
print('cross-view RLE byte-identical: %d/%d (%.1f%%) | symbol-set Jaccard median %.3f'
      % (same, tot, 100 * same / tot, np.median(jac)))
print('views per execution:', dict(sorted(Counter(len(v) for v in groups.values()).items())))
print('per-activity executions:', dict(Counter(labels).most_common()))

# D_G from the FULL X (as 06m does): the alphabet geometry should not depend on our subset,
# and building from a subset can silently shrink G if a top-id symbol is absent.
D_G = build_action_seg_ground_cost(NAME, X=X_raw, kind='cooccurrence')
if D_G.shape[0] != G:
    print('NOTE: G reconciled %d -> %d (loader len(mapping) vs ground-cost max+1)' % (G, D_G.shape[0]))
    G = D_G.shape[0]
print('G =', G, '| D_G', D_G.shape)

## 2. The screen — every cached real task

This table is the deliverable, regardless of what the order-null later says.

In [ ]:
scr_bf = headroom_table(X, labels, G, D_G, name='breakfast_activity')
print('Breakfast: %d activity pairs | bands: %s'
      % (len(scr_bf), dict(Counter(scr_bf['headroom_band']))))
print('hist_auc: min=%.3f  median=%.3f  max=%.3f'
      % (scr_bf.hist_auc.min(), scr_bf.hist_auc.median(), scr_bf.hist_auc.max()))
print('\nthe 8 pairs with the MOST shared vocabulary (best a priori candidates):')
display(scr_bf.sort_values('vocab_jaccard', ascending=False)
        .head(8)[['comparison', 'n', 'vocab_jaccard', 'n_shared', 'hist_auc', 'headroom_band']]
        .round(3))

In [ ]:
screen = [scr_bf]

# SDS2 G=28 -- the cohort whose 0/15 null the paper reports. Did IT have headroom?
from smartflat.features.symbolic_barycenter.vocab import load_g28_cohort, build_g28_ground_cost
df28, X28, l28 = load_g28_cohort()
D28 = build_g28_ground_cost()
scr_sds2 = headroom_table(X28, l28, D28.shape[0], D28, name='sds2_g28')
screen.append(scr_sds2)
print('SDS2 G=28 (N=%d, %s)' % (len(X28), dict(Counter(l28))))
display(scr_sds2[['comparison', 'n', 'hist_auc', 'vocab_jaccard', 'headroom_band']].round(3))

# GTEA -- 7 classes x exactly 4 videos: below the 5-fold minimum, so the screen and the null
# both return zero rows. Recorded as untestable rather than silently absent.
mg, Xg, lg, Gg = load_action_seg('gtea')
scr_gtea = headroom_table(Xg, lg, Gg, name='gtea')
screen.append(scr_gtea)
print('\nGTEA: %d classes x min %d videos < n_folds=5 -> %d screenable pairs (UNTESTABLE)'
      % (len(set(lg)), min(Counter(lg).values()), len(scr_gtea)))
print('50Salads: loader gives 1 activity class -> 0 activity pairs; revisited as a trial-split control in §6')

screen = pd.concat(screen, ignore_index=True)
screen.to_csv(os.path.join(OUT, 'screen.csv'), index=False)
print('\n=== SCREEN SUMMARY (band counts per dataset) ===')
display(screen.groupby(['dataset', 'headroom_band']).size().unstack(fill_value=0))

## 3. Constructing a frequency-controlled task

Every natural Breakfast contrast is saturated, so no *naturally* frequency-controlled activity task exists
here. We construct one: **drop each pair's class-exclusive marker symbols and keep only the actions both
classes perform**, then ask the narrower, honest question:

> *Given only the actions the two recipes have in common, does the **order** of those actions distinguish them?*

The retained symbols keep their relative order, so the output is a genuine subsequence of the real
annotation — every sequence is real, nothing is synthesised. The restriction is **label-derived**, but in
the *anti*-leakage direction: it **removes** perfectly class-diagnostic features rather than keeping them.
And, like the screen, it is a function of shuffle-invariant quantities (a vocabulary is the support of a
multiset), so the null stays conditionally valid.

**Disclosures.** `hist_auc` below is the frequency AUC *of the constructed sub-task*, not of Breakfast.
`collapse_runs=False`: deleting a marker between two runs of the same symbol leaves `a a`, which preserves
the multiset the null holds fixed (collapsing would change it) and keeps the 2×2 grid non-degenerate. Both
branches were checked and land in the same band. `min_len=2` drops restricted sequences with no transition;
per-class drop counts are printed.

In [ ]:
CANDIDATES = [('friedegg', 'pancake'), ('friedegg', 'scrambledegg'),
              ('pancake', 'scrambledegg'), ('coffee', 'tea'), ('coffee', 'milk')]
inv = {v: k for k, v in read_mapping(os.path.join(default_root(NAME), 'mapping.txt'),
                                     DATASETS[NAME]['background']).items()}
TASKS, task_scr = {}, []
for pair in CANDIDATES:
    tag = 'bf_shared_%s_vs_%s' % pair
    try:
        Xr, lr, Gc, Dc, info = restrict_to_shared_vocabulary(
            X, labels, pair, D_G=D_G, collapse_runs=False, min_len=2)
    except ValueError as e:
        print('SKIP %-32s %s' % (tag, e)); continue
    h = headroom_table(Xr, lr, Gc, Dc, name=tag)
    if len(h) == 0:
        print('SKIP %-32s too few sequences survive for 5-fold (n=%d)' % (tag, len(Xr))); continue
    TASKS[tag] = dict(X=Xr, labels=lr, G=Gc, D_G=Dc, info=info, pair=pair,
                      band=h.iloc[0]['headroom_band'], hist_auc=h.iloc[0]['hist_auc'])
    task_scr.append(h)
    r = h.iloc[0]
    print('%-32s n=%3d G=%d hist_auc=%.3f band=%-9s len(med=%d,max=%d) dropped=%s'
          % (tag, len(Xr), Gc, r['hist_auc'], r['headroom_band'],
             int(np.median(info['lengths'])), int(info['lengths'].max()), dict(info['n_dropped'])))
    print('%s shared: %s' % (' ' * 4, [inv[s] for s in info['shared']]))
    for cls in pair:
        ex = [q for q, L in zip(Xr, lr) if L == cls][:2]
        print('%s %-14s e.g. %s' % (' ' * 4, cls,
              ' | '.join('->'.join(inv[info['shared'][t]] for t in e) for e in ex)))

task_scr = pd.concat(task_scr, ignore_index=True)
screen = pd.concat([screen, task_scr], ignore_index=True)
screen.to_csv(os.path.join(OUT, 'screen.csv'), index=False)
display(task_scr[['comparison', 'dataset', 'n', 'hist_auc', 'n_shared', 'headroom_band']].round(3))

## 4. Pre-registration — frozen before any null is run

- **Primary tasks:** the constructed tasks in the `sweet` band. `floor`-band tasks are **not tests** — they
  are reported as controls, since a null on them cannot distinguish "order doesn't help" from "these classes
  are exchangeable".
- **Primary combo (one):** `run_transition × runlength` — the dwell-invariant feature against the
  dwell-preserving shuffle, i.e. the cleanest pure-sequencing probe. **Why only one:**
  `p_perm = (1 + #{null ≥ intact}) / (1 + n_shuffles)`, so at 200 shuffles the smallest attainable p is
  `1/201 = 0.00498`, while BH at α=0.05 over m=16 cells would require `p ≤ 0.05/16 = 0.0031` at rank 1 —
  **unreachable**, so a lone true positive could never be declared. One combo (m = #sweet tasks) at
  `n_shuffles=1000` (`p_min = 0.000999`) keeps the test declarable.
- **Family:** the primary rows only. BH `fdr_bh` over `p_perm`. The full 2×2 grid at 200 shuffles is a
  **secondary/exploratory** family, BH'd within itself, never used for the headline. Screen rows are
  descriptive, not tests. The frozen 06m/SDS2 cells are already reported and are not retro-pooled.
- **Decision rule (all three required):** `BH(p_perm) < 0.05` **AND** `auc_intact > hist_auc` **AND** the
  shuffle-free incremental probe's `delta_ci_low > 0`.
- **Why the extra two conditions.** The shuffle null tests *"order is uniform given the multiset"*, which is
  strictly stronger than what we care about, *"order is independent of the label given the multiset"*. Real
  sequences violate the former massively while possibly satisfying the latter — and then shuffling **blurs**
  the frequency signal (a sharp structured transition matrix degrades into a noisy one) and `delta_auc` goes
  positive with **zero** order-label association. So the null is **anti-conservative in exactly this
  regime**. `auc_intact > hist_auc` is the diagnostic that catches it, and the incremental probe (whose
  `both` feature set *nests* `hist` on identical folds) is structurally immune to it. Note this bias only
  threatens *positives*: every existing null (SDS2 0/15, Breakfast 0/60) is, if anything, **strengthened**.
- **Calibration gate (§5, run first):** if the decoy's empirical type-I rate is materially above ~10%, this
  notebook reports a **null-with-caveat and makes no positive claim**, whatever `p_perm` says.

## 5. Calibration decoy — the probe's type-I rate at this operating point

`pooled_markov_surrogate` keeps every sequence's multiset **bit-identical** (so the frequency–label
relationship, and `hist_auc`, are fully preserved) while resampling its arrangement from a **class-pooled**
first-order model — so order carries **provably no label information** by construction. The *unmodified*
null must therefore fail to reject. The rate at which it *does* reject is its empirical type-I rate on this
dataset's own structure. This runs **before** the real null so it cannot be tuned to the answer.

In [ ]:
PRIMARY = (('run_transition', 'runlength'),)
N_DECOY, N_SH_DECOY, ALPHA = 10, 200, 0.05
PRIMARY_TASKS = [t for t, d in TASKS.items() if d['band'] == 'sweet']
print('primary (sweet-band) tasks:', PRIMARY_TASKS)
print('control (floor/saturated) tasks:', [t for t in TASKS if t not in PRIMARY_TASKS])

cal = []
for tag in PRIMARY_TASKS:
    d = TASKS[tag]
    for k in range(N_DECOY):
        S = pooled_markov_surrogate(d['X'], d['G'], random_state=k)
        r = order_information(S, d['labels'], d['G'], feature='run_transition',
                              shuffle='runlength', n_shuffles=N_SH_DECOY,
                              random_state=42).iloc[0]
        cal.append({'task': tag, 'replicate': k, 'auc_intact': r['auc_intact'],
                    'auc_null_mean': r['auc_null_mean'], 'delta_auc': r['delta_auc'],
                    'p_perm': r['p_perm'], 'order_helps': bool(r['order_helps'])})
cal = pd.DataFrame(cal)
cal.to_csv(os.path.join(OUT, 'calibration.csv'), index=False)
TYPE1 = cal.groupby('task')['order_helps'].mean()
print('\n=== empirical type-I rate (nominal 0.025; %d decoys x %d shuffles) ==='
      % (N_DECOY, N_SH_DECOY))
display(pd.DataFrame({'type_I_rate': TYPE1.round(3),
                      'mean_delta_auc': cal.groupby('task')['delta_auc'].mean().round(4)}))
print('CALIBRATION GATE: %s' % ('PASS - positives interpretable' if TYPE1.max() <= 0.10 else
                                'FAIL - probe anti-conservative here; report null-with-caveat only'))

## 6. The order-null — sweet-band tasks only

In [ ]:
prim = []
for tag in PRIMARY_TASKS:
    d = TASKS[tag]
    res = run_generalization_suite(d['X'], d['labels'], d['G'], d['D_G'], name=tag,
                                   run_quality=False, order_grid=PRIMARY, n_shuffles=1000)
    prim.append(res['order'])
prim = pd.concat(prim, ignore_index=True)
prim['hist_auc'] = [TASKS[t]['hist_auc'] for t in prim['dataset']]
prim['beats_frequency'] = prim['auc_intact'] > prim['hist_auc']

from statsmodels.stats.multitest import multipletests
rej, p_bh, _, _ = multipletests(prim['p_perm'].to_numpy(), alpha=ALPHA, method='fdr_bh')
prim['p_bh'], prim['reject_bh'] = p_bh, rej
prim.to_csv(os.path.join(OUT, 'order_null.csv'), index=False)
print('=== PRIMARY family (m=%d, one pre-registered combo, 1000 shuffles, BH alpha=%.2f) ==='
      % (len(prim), ALPHA))
display(prim[['dataset', 'comparison', 'feature', 'shuffle', 'hist_auc', 'auc_intact',
              'auc_null_mean', 'delta_auc', 'p_perm', 'p_bh', 'reject_bh',
              'beats_frequency']].round(4))

In [ ]:
# Secondary/exploratory: the full 2x2 grid at 200 shuffles, on EVERY constructed task
# (incl. floor-band controls). BH within this family only; never used for the headline.
sec = []
for tag, d in TASKS.items():
    res = run_generalization_suite(d['X'], d['labels'], d['G'], d['D_G'], name=tag,
                                   run_quality=False, n_shuffles=200)
    sec.append(res['order'])
sec = pd.concat(sec, ignore_index=True)
sec['band'] = [TASKS[t]['band'] for t in sec['dataset']]
sec['hist_auc'] = [TASKS[t]['hist_auc'] for t in sec['dataset']]
sec['beats_frequency'] = sec['auc_intact'] > sec['hist_auc']
rej, p_bh, _, _ = multipletests(sec['p_perm'].to_numpy(), alpha=ALPHA, method='fdr_bh')
sec['p_bh'], sec['reject_bh'] = p_bh, rej
sec.to_csv(os.path.join(OUT, 'order_null_secondary.csv'), index=False)
print('=== SECONDARY family (exploratory): %d cells, %d reject after BH ==='
      % (len(sec), int(sec['reject_bh'].sum())))
display(sec[['dataset', 'band', 'feature', 'shuffle', 'hist_auc', 'auc_intact',
             'delta_auc', 'p_perm', 'p_bh', 'reject_bh', 'beats_frequency']].round(4))

## 7. Confirmatory probe — shuffle-free

`evaluate_incremental_ordering` trains on `hist` vs `both = [hist ⊕ transition]`, paired on identical folds.
Because `both` **nests** `hist`, adding order features cannot help unless order carries information the
histogram lacks — so this probe is structurally immune to the blur artifact that makes the shuffle null
anti-conservative. **Its own caveat:** its bootstrap/Wilcoxon treat CV folds as independent, which they are
not (repeated folds share training data), so its intervals are anti-conservative in the *other* direction.
Neither probe is trustworthy alone — the pre-registered rule requires **both** to fire, and effect sizes
are read in preference to p-values.

In [ ]:
inc = []
for tag in PRIMARY_TASKS:
    d = TASKS[tag]
    # ragged -> object array: evaluate_incremental_ordering does a bare np.asarray, which
    # raises on a ragged list under numpy>=1.24.
    folds, summ = evaluate_incremental_ordering(
        np.asarray(d['X'], dtype=object), d['labels'], d['G'],
        classifiers=('logreg',), n_repeats=3, n_folds=5, random_state=42)
    inc.append(summ.assign(dataset=tag))
inc = pd.concat(inc, ignore_index=True)
inc.to_csv(os.path.join(OUT, 'incremental.csv'), index=False)
display(inc[['dataset', 'comparison', 'mean_auc_hist', 'mean_auc_both', 'mean_delta',
             'delta_ci_low', 'delta_ci_high', 'wilcoxon_p']].round(4))

## 8. Real-data negative control — 50Salads trial-1 vs trial-2

25 subjects each prepare the **same recipe twice**, so the two classes share a vocabulary *by construction* —
no restriction needed. This is a **control, not a test**: we have strong prior reason to believe the two
attempts are exchangeable, so the screen should place it at the `floor` and the null should not fire. If it
does, the probe is broken. The comparison is paired (every subject is in both classes), which the harness's
non-grouped CV does not model — that is conservative, not leaky: subject identity is exactly uninformative
about the label, so it cannot be exploited to inflate AUC. (Its sub-0.5 `hist_auc` is the signature of that
pairing pulling predictions toward the wrong label.)

In [ ]:
m5, X5, l5, G5 = load_action_seg('50salads')
D5 = build_action_seg_ground_cost('50salads', X=X5, kind='cooccurrence')
G5 = D5.shape[0]
y5 = trial_labels(m5, '50salads')
print('50Salads: N=%d  %s  G=%d' % (len(X5), dict(Counter(y5)), G5))
scr5 = headroom_table(X5, y5, G5, D5, name='50salads_trial')
display(scr5[['comparison', 'n', 'hist_auc', 'vocab_jaccard', 'headroom_band']].round(3))

res5 = run_generalization_suite(X5, y5, G5, D5, name='50salads_trial',
                                run_quality=False, n_shuffles=200)
ord5 = res5['order']
print('control: order_helps %d/%d (expected 0 -- the classes are exchangeable by design)'
      % (int(ord5['order_helps'].sum()), len(ord5)))
display(ord5[['comparison', 'feature', 'shuffle', 'auc_intact', 'auc_null_mean',
              'delta_auc', 'p_perm', 'order_helps']].round(4))
screen = pd.concat([screen, scr5], ignore_index=True)
screen.to_csv(os.path.join(OUT, 'screen.csv'), index=False)
ord5.to_csv(os.path.join(OUT, 'negative_control_50salads.csv'), index=False)

## 9. Mechanistic decode

**Descriptive, not a test** — fit in-sample on the full restricted task purely to read *what* the order
features key on. An artifact of the blur kind produces no interpretable, multiset-invariant signature; a
real ordering effect does.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from smartflat.features.symbolic_barycenter.order_evaluation import run_transition_features

for tag in PRIMARY_TASKS:
    d = TASKS[tag]; a, b = d['pair']; sh = d['info']['shared']
    F = run_transition_features(d['X'], d['G'])
    y = (np.asarray(d['labels'], dtype=object) == b).astype(int)
    pipe = Pipeline([('s', StandardScaler()),
                     ('c', LogisticRegression(penalty='l2', solver='liblinear', C=1.0,
                                              max_iter=1000))]).fit(F, y)
    coef = pipe.named_steps['c'].coef_[0]
    print('\n=== %s   (+ favours %r, - favours %r) ===' % (tag, b, a))
    for j in np.argsort(-np.abs(coef))[:6]:
        u, v = divmod(j, d['G'])
        if F[:, j].sum() == 0:
            continue
        print('   %-42s coef=%+.3f' % ('%s -> %s' % (inv[sh[u]], inv[sh[v]]), coef[j]))

## 10. Verdict

In [ ]:
rows = []
for tag in TASKS:
    d = TASKS[tag]
    p = prim[prim['dataset'] == tag]
    i = inc[inc['dataset'] == tag]
    rows.append({
        'task': tag, 'n': len(d['X']), 'band': d['band'],
        'hist_auc': round(d['hist_auc'], 3),
        'auc_intact': round(float(p['auc_intact'].iloc[0]), 3) if len(p) else np.nan,
        'delta_auc': round(float(p['delta_auc'].iloc[0]), 3) if len(p) else np.nan,
        'p_bh': round(float(p['p_bh'].iloc[0]), 4) if len(p) else np.nan,
        'beats_freq': bool(p['beats_frequency'].iloc[0]) if len(p) else None,
        'incr_delta': round(float(i['mean_delta'].iloc[0]), 3) if len(i) else np.nan,
        'incr_ci_low': round(float(i['delta_ci_low'].iloc[0]), 3) if len(i) else np.nan,
        'type_I': round(float(TYPE1.get(tag, np.nan)), 3),
        'median_len': int(np.median(d['info']['lengths'])),
    })
verdict = pd.DataFrame(rows)
# the pre-registered rule: all three conditions, and only for sweet-band tasks
verdict['ORDER_HELPS'] = [
    bool(r['band'] == 'sweet' and r['p_bh'] == r['p_bh'] and r['p_bh'] < ALPHA
         and r['beats_freq'] and r['incr_ci_low'] == r['incr_ci_low'] and r['incr_ci_low'] > 0)
    for _, r in verdict.iterrows()]
verdict.to_csv(os.path.join(OUT, 'verdict.csv'), index=False)
display(verdict)

print('\n--- frozen 06m artifacts (must be byte-identical) ---')
for p_, h in FROZEN.items():
    now = hashlib.sha256(open(p_, 'rb').read()).hexdigest()
    print('  %-16s %s' % (os.path.basename(p_), 'UNCHANGED' if now == h else 'CHANGED !!!'))
    assert now == h, '06m artifact %s was modified' % p_
print('\nwrote:', sorted(os.listdir(OUT)))

### Reading of record

**The headline is the screen.** It separates the nulls that were *evidence* from the nulls that were
*arithmetic*:

- **SDS2 G=28 has genuine headroom** (`hist_auc` ≈ 0.70–0.85 across the three frozen comparisons). Its
  **0/15 was a fair test** — real evidence that order adds nothing beyond frequency on that cohort.
- **Breakfast's activity contrasts have none** — **45/45** pairs at `hist_auc = 1.000`, even after
  deduplicating to 503 distinct executions, and even for the most vocabulary-sharing pair
  (`friedegg`/`scrambledegg`, Jaccard 0.583). 06m's **0/60 was vacuous**: the ceiling left the null no room.
  The honest denominator is smaller still — the loader run-length-encodes, so `transition_features` and
  `run_transition_features` are *identical* on its output and the 2×2 grid collapses to **one** distinct
  test: 0/60 is 15 distinct tests replicated 4×.
- **GTEA is untestable** (7 classes × 4 videos < the 5-fold minimum), and **50Salads' trial split sits at the
  floor** — a valid negative control, not a test.

So: **no naturally frequency-controlled real task exists in this corpus.** That is itself the finding, and
it is why the coarse-activity generalisation datasets cannot adjudicate "does order help".

**The constructed sub-result is tagged and scope-limited.** See the verdict table above for what the
restricted contrasts return. Whatever the sign, the supportable claim is narrow: it concerns *the relative
order of a handful of shared actions in one constructed contrast*, on sequences whose median restricted
length is 2–4 symbols. It does **not** support "order matters in procedural activity", and it does not
overturn SDS2's negative — which the screen has now shown was a fair test all along.

**For the paper (§7.2 / §1.6).** The negative ordering result stands and is *strengthened*: the shuffle null
is anti-conservative when frequency signal is present, so a null obtained from it is conservative evidence.
Cite the screen when claiming external validity — it is what licenses "SDS2's 0/15 means something", and it
retires the Breakfast 0/60 as evidence for anything.